# K-Means Clustering

This performs and visualises manual k-means clustering, as a simplified version of what something like scikit-learn's KMeans would perform.

In [133]:
# Imports
import time
from typing import Generator

from IPython.display import HTML, Image
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

In [134]:
clusters = 10
# Additional frames to add after clustering has converged
stable_frames = 3
# Max iterations of k-means to run
max_iterations = 100
epsilon = 0.000001
# Seeds to use for random generator for each run
seeds = [2, 3, 6, 7, 9, 19]

# Variables for point generation
distance_from_cluster = 0.15
number_of_points_near_clusters = 100
number_of_random_points = 100

def add_scatter(cluster: int, data: np.ndarray, update: bool = False, **kwargs) -> None:
    """ Add or update scatter plot for a given cluster (or cluster centroid, which is offset by `clusters`) """
    x, y = data[:, 0], data[:, 1]
    if update:
        scatter_plots[cluster].set_offsets(np.stack([x, y]).T)
    else:
        scatter_plots.append(ax.scatter(x, y, **kwargs))

def update_plot(frame: int) -> list:
    """ Perform 1 step of k-means clustering to update the scatter plot """
    global last
    global assignments
    global centers

    # Set centers to mean of cluster points on second frame and beyond
    if frame != 0:
        last = centers.copy()
        centers = np.array([points[assignments == i].mean(axis=0) for i in range(clusters)])

    # Assign each point to the nearest cluster
    # Convert the 2D points to (M, 1, 2) and centers to (1, N, 2), then get the cross-join of (M, N, 2) square distances
    square_distances = np.sum((points[:, None, :] - centers[None, :, :]) ** 2, axis=2)
    # The assignment for each point is the closest center (i.e. the one which minimises square distance)
    assignments = square_distances.argmin(axis=1)

    update = bool(scatter_plots)

    # Reset to default color cycle for consistent cluster coloring
    ax.set_prop_cycle(None)

    # Plot points for each cluster
    for cluster in range(clusters):
        add_scatter(cluster, points[assignments == cluster], update=update)

    # Reset to default color cycle for consistent cluster coloring
    ax.set_prop_cycle(None)

    # Plot cluster centroids
    for cluster in range(clusters):
        add_scatter(clusters + cluster, centers[cluster:cluster+1], update=update, s=200)

    return scatter_plots

for seed in seeds:
    rng = np.random.default_rng(seed)

    # Generate some clusters centroids to focus points around
    true_clusters = rng.random((clusters, 2))

    # Generate points within a square around the clusters
    # Generate 3 values for each point: the cluster index, and the x and y offsets from that cluster
    points = rng.random((number_of_points_near_clusters, 3))
    points = true_clusters[(points[:, 0] * clusters).astype(int)] + (points[:, 1:] - 0.5) * distance_from_cluster

    # Add purely random points
    points = np.concat([points, rng.random((number_of_random_points, 2))])

    # Select random points as start of k-means
    centers = rng.choice(points, clusters, replace=False, axis=0)

    assignments = np.zeros(len(points))

    def frame_generator() -> Generator[int]:
        """ Dynamic generator for frames, which stops when k-mean converges (or when the max number of iterations are reached) """
        for i in range(max_iterations):
            if last is not None and ((np.array(last) - np.array(centers))**2).sum(axis=1).max() < epsilon:
                break
            yield i
        # Add additional frames after clustering has converged
        yield from range(i, i + stable_frames)

    fig, ax = plt.subplots()
    scatter_plots = []
    last = None

    # Generate and save animations of k-means
    ani = animation.FuncAnimation(fig=fig, func=update_plot, frames=frame_generator, interval=200, cache_frame_data=False)
    plt.close(fig)  # don't display empty figure
    ani.save(filename=f'intermediate-files/cluster{seed}.gif', writer='pillow')

In [135]:
# Display clustering animations. Use `?time` to circumvent image caching
for row in np.array_split(seeds, 2):
    display(HTML(
        '<div style="display: flex; justify-content: space-around;">' +
        "".join(f'<img src="intermediate-files/cluster{i}.gif?{time.time()}" style="width:30%;"/>' for i in row) +
        '</div>'
    ))